In [1]:
import cv2
import os
import torch
from facenet_pytorch import MTCNN, InceptionResnetV1

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

mtcnn = MTCNN(keep_all=True, device=device)
resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

known_embeddings = []
known_names = []
persons_dir = "persons"

CACHE_FILE = "embeddings_cache.pt"
if os.path.exists(CACHE_FILE):
    print(f"Loading embeddings directly from {CACHE_FILE}...")
    data = torch.load(CACHE_FILE, map_location=device)
    known_embeddings = data['embeddings']
    known_names = data['names']
    print(f"Loaded {len(known_names)} faces from cache: {known_names}")

else:
    print("No cache found. Loading known faces from images...")
    for person_name in os.listdir(persons_dir):
        person_path = os.path.join(persons_dir, person_name)
        if os.path.isdir(person_path):
            for img_name in os.listdir(person_path):
                img_path = os.path.join(person_path, img_name)
                try:
                    img_bgr = cv2.imread(img_path)
                    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

                    face_tensor = mtcnn(img_rgb)
                    
                    if face_tensor is not None:
                        face_tensor = face_tensor[0].unsqueeze(0).to(device)
                        
                        with torch.no_grad():
                            embedding = resnet(face_tensor)
                        
                        known_embeddings.append(embedding)
                        known_names.append(person_name)
                        break
                    else:
                        print(f"No face detected in {img_path}")
                except Exception as e:
                    print(f"Error loading image {img_path}: {e}")

    print(f"Saving {len(known_names)} faces to {CACHE_FILE}...")
    torch.save({
        'embeddings': known_embeddings,
        'names': known_names
    }, CACHE_FILE)
    print("Saved successfully!")

c:\Users\Mahmoud Refaat\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


No cache found. Loading known faces from images...
Saving 3 faces to embeddings_cache.pt...
Saved successfully!


In [2]:
import cv2
import os
import torch
import tkinter as tk
from tkinter import filedialog, messagebox
from PIL import Image, ImageTk
from facenet_pytorch import MTCNN, InceptionResnetV1

class FaceRecognitionApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Face Recognition App")
        self.root.geometry("800x600")
        self.last_drawn_data = []
        

        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.mtcnn = MTCNN(keep_all=True, device=self.device)
        self.resnet = InceptionResnetV1(pretrained='vggface2').eval().to(self.device)
        
        self.known_embeddings = []
        self.known_names = []
        self.EUCLIDEAN_THRESHOLD = 0.9
        self.scaling_factor = 0.5
        
        self.load_embeddings_cache()


        self.cap = None
        self.video_loop_id = None
        self.is_running = False


        self.menu_frame = tk.Frame(self.root)
        self.menu_frame.pack(expand=True)
        
        title_label = tk.Label(self.menu_frame, text="Face Recognition System", font=("Arial", 20, "bold"))
        title_label.pack(pady=20)

        btn_image = tk.Button(self.menu_frame, text="Choose Image", font=("Arial", 14), width=20, command=self.open_image)
        btn_image.pack(pady=10)

        btn_video = tk.Button(self.menu_frame, text="Choose Video", font=("Arial", 14), width=20, command=self.open_video)
        btn_video.pack(pady=10)

        btn_webcam = tk.Button(self.menu_frame, text="Open Webcam", font=("Arial", 14), width=20, command=self.open_webcam)
        btn_webcam.pack(pady=10)

        self.display_frame = tk.Frame(self.root)
        
        self.btn_stop = tk.Button(self.display_frame, text="Stop / Return", font=("Arial", 12), bg="red", fg="white", command=self.stop_and_return)
        self.btn_stop.pack(pady=10)

        self.canvas_label = tk.Label(self.display_frame)
        self.canvas_label.pack(expand=True)

    def load_embeddings_cache(self):
        CACHE_FILE = "embeddings_cache.pt"
        if os.path.exists(CACHE_FILE):
            print("Loading embeddings from cache...")
            data = torch.load(CACHE_FILE, map_location=self.device)
            self.known_embeddings = data['embeddings']
            self.known_names = data['names']
        else:
            messagebox.showwarning("Warning", "File embeddings_cache.pt not found. Please run the face extraction code first.")

    def process_frame(self, frame):
        small_frame = cv2.resize(frame, (0, 0), fx=self.scaling_factor, fy=self.scaling_factor)
        frame_rgb = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)

        boxes, probs = self.mtcnn.detect(frame_rgb)
        current_data = []

        if boxes is not None:
            faces_tensors = self.mtcnn.extract(frame_rgb, boxes, save_path=None) # list of tensors

            for i, box in enumerate(boxes):
                if probs[i] > 0.9 and faces_tensors is not None:
                    x, y, x2, y2 = [int(b / self.scaling_factor) for b in box]

                    face_tensor = faces_tensors[i].unsqueeze(0).to(self.device)
                    with torch.no_grad():
                        face_embedding = self.resnet(face_tensor)

                    min_dist = float('inf')
                    name = "Unknown"

                    if self.known_embeddings:
                        distances = [torch.dist(face_embedding, known_emb).item() for known_emb in self.known_embeddings]
                        min_dist = min(distances)

                        if min_dist < self.EUCLIDEAN_THRESHOLD:
                            idx = distances.index(min_dist)
                            name = self.known_names[idx]

                    current_data.append({"box": (x, y, x2, y2), "name": name, "dist": min_dist})
            
        self.last_drawn_data = current_data

        for data in self.last_drawn_data:
            x, y, x2, y2 = data["box"]
            name = data["name"]
            min_dist = data["dist"]

            color = (0, 255, 0) if name != "Unknown" else (0, 0, 255)
            cv2.rectangle(frame, (x, y), (x2, y2), color, 2)
            
            display_text = f"{name} ({min_dist:.2f})" if name != "Unknown" else "Unknown"
            y_text = y - 10 if y - 10 > 10 else y + 10
            cv2.putText(frame, display_text, (x, y_text), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
        
        return frame

    def display_cv2_image(self, cv2_img):
        cv2_img_rgb = cv2.cvtColor(cv2_img, cv2.COLOR_BGR2RGB)
        pil_img = Image.fromarray(cv2_img_rgb)
        
        pil_img.thumbnail((700, 500)) 
        
        imgtk = ImageTk.PhotoImage(image=pil_img)
        self.canvas_label.imgtk = imgtk
        self.canvas_label.configure(image=imgtk)

    def switch_to_display(self):
        self.menu_frame.pack_forget()
        self.display_frame.pack(fill=tk.BOTH, expand=True)
        self.is_running = True

    def stop_and_return(self):
        self.is_running = False
        if self.cap is not None:
            self.cap.release()
            self.cap = None
        
        if self.video_loop_id is not None:
            self.root.after_cancel(self.video_loop_id)
            self.video_loop_id = None
            
        self.canvas_label.configure(image='')
        self.display_frame.pack_forget()
        self.menu_frame.pack(expand=True)

    def open_image(self):
        file_path = filedialog.askopenfilename(title="Choose Image", filetypes=[("Image files", "*.jpg *.jpeg *.png")])
        if file_path:
            self.switch_to_display()
            img = cv2.imread(file_path)
            if img is not None:
                processed_img = self.process_frame(img)
                self.display_cv2_image(processed_img)

    def open_video(self):
        file_path = filedialog.askopenfilename(title="Choose Video", filetypes=[("Video files", "*.mp4 *.avi *.mkv")])
        if file_path:
            self.switch_to_display()
            self.cap = cv2.VideoCapture(file_path)
            self.video_loop()

    def open_webcam(self):
        self.switch_to_display()
        self.cap = cv2.VideoCapture(0)
        self.video_loop()

    def video_loop(self):
        if self.is_running and self.cap is not None:
            ret, frame = self.cap.read()
            if ret:
                processed_frame = self.process_frame(frame)
                self.display_cv2_image(processed_frame)
                self.video_loop_id = self.root.after(10, self.video_loop)
            else:
                messagebox.showinfo("Finished", "Video playback finished")
                self.stop_and_return()

if __name__ == "__main__":
    root = tk.Tk()
    app = FaceRecognitionApp(root)
    root.mainloop()

Loading embeddings from cache...
